# Data compilation script
Produces aggregated results over multiple training seeds.

Produces: `all_channels_results_qc_clean.csv`

In [1]:
import pandas as pd
from scipy.stats import pearsonr
import numpy as np
from functools import reduce
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import anndata
from src.eval import calculate_results_per_batch
import torch

out_dir = './results'
fig_dir = f'{out_dir}/Figures'

In [2]:
whitelist_df = pd.read_csv(f'../input_files/cid_whitelist.csv')
toremove_df = pd.read_csv(f'../input_files/cid_nuc_toremove.csv')
whitelist_ids = np.array(whitelist_df.columns)
remove_cell_ids = np.array(toremove_df.columns)

#### Aggregate results

In [9]:
seeds = [0, 1, 2, 3, 4]

model_paths = [
    'fucci_final_488-561-seg-bg-random/mlp_fucci_angle_model',
    'fucci_final_405-bg-random_resnet',
    'fucci_final_seg-bg-random_resnet'
]

for model_path in model_paths:
    print(model_path)
    for seed in seeds:
        path = out_dir + '/' + model_path + '/seed_{}/test_preds.h5ad'
        test_adata = anndata.read_h5ad(path.format(seed))
        keep_idxs = np.intersect1d(test_adata.obs.index, whitelist_ids)
        test_adata_ = test_adata[keep_idxs]
        print(test_adata_.shape, test_adata.shape)
        results_df = calculate_results_per_batch(test_adata_, test_adata_.layers['pred'], test_adata_.X)
        results_df.to_csv(out_dir + '/' + model_path + '/seed_{}/'.format(seed) + 'test_results_qc.csv')

fucci_final_488-561-seg-bg-random/mlp_fucci_angle_model
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_405-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_seg-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)


/home/bues/iris_DataManagement/geneexp_data/ds_tools/IRIS/2_Cellcycle/DeepLearning_pipeline/src/eval.py:26: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  corr_nonzero, pval_nonzero = pearsonr(gts[m, idx], preds[m, idx])


In [10]:
def calculate_consensus(path_template, seeds, alpha=0.05, metric='avg_Pearson'):
    results_dfs = []
    pvals_dfs = []
    for seed in seeds:
        # print(path_template.format(seed=seed))
        results_df = pd.read_csv(path_template.format(seed), index_col='genes')
        results_df[f'{metric}_{seed}'] = results_df[metric]

        # Combine p-values across batches
        test_batches = [s.split('_')[1] for s in results_df.columns if s.startswith('Pearson_')]
        batch_pval_columns = [f'pvals_{s}' for s in test_batches]
        pvalues = []
        for g, row in results_df[batch_pval_columns].iterrows():
            combined_pval = combine_pvalues(row.values, method='fisher')[1]
            pvalues.append(combined_pval)
        results_df[f'{metric}_pval'] = pvalues
        pvals_dfs.append(results_df[f'{metric}_pval'])
        results_dfs.append(results_df[f'{metric}_{seed}'])
    
    # Average results
    merged_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), results_dfs)
    merged_df[metric] = merged_df.values.mean(axis=-1)

    # Combine p-values across seeds
    genes = merged_df.index
    combined_pvals_df = pd.DataFrame(index=genes, columns=['combined_pval'])
    for g in genes:
        pvals = [df.loc[g] for df in pvals_dfs]
        combined_pvals_df.loc[g] = combine_pvalues(pvals, method='fisher')[1]
    merged_df[f'{metric}_pval'] = combined_pvals_df[f'combined_pval']
    merged_df[f'{metric}_fdr'] =  multipletests(combined_pvals_df[f'combined_pval'], alpha=alpha, method='fdr_bh')[1]
    # false_discovery_control(combined_pvals_df[f'combined_pval'])
    return merged_df

In [11]:
seeds = [0, 1, 2, 3, 4]
filename = 'test_results_qc.csv'

fucci405_merged_df = calculate_consensus(out_dir +'/fucci_final_405-bg-random_resnet/seed_{}/'+filename, seeds)
fucci405_merged_df['channel'] = 'Hoechst'

fucci_df = calculate_consensus(out_dir + '/fucci_final_488-561-seg-bg-random/mlp_fucci_angle_model/seed_{}/'+filename, seeds, metric='avg_Pearson')
fucci_df['channel'] = 'FUCCI angle'

fucci_seg_merged_df = calculate_consensus(out_dir + '/fucci_final_seg-bg-random_resnet/seed_{}/'+filename, seeds, metric='avg_Pearson')
fucci_seg_merged_df['channel'] = 'Segmentation'

all_merged_df = pd.concat([fucci405_merged_df, fucci_df,fucci_seg_merged_df], axis=0)
all_merged_df = all_merged_df[['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr', 'channel']]
print(all_merged_df)

          avg_Pearson avg_Pearson_pval avg_Pearson_fdr       channel
genes                                                               
Top2a        0.349018              0.0             0.0       Hoechst
Prc1         0.316526              0.0             0.0       Hoechst
Spc25        0.265249              0.0             0.0       Hoechst
Nusap1       0.251010              0.0             0.0       Hoechst
Kif20b       0.224234              0.0             0.0       Hoechst
...               ...              ...             ...           ...
Serpine3    -0.042550         0.264453        0.433503  Segmentation
Cd180        0.006069         0.202571        0.351476  Segmentation
Mtfmt       -0.028677         0.031097        0.079307  Segmentation
Fam229b     -0.001564         0.187171        0.333091  Segmentation
Gm10240     -0.028606         0.031864        0.080818  Segmentation

[4185 rows x 4 columns]


In [12]:
df_wide = all_merged_df.reset_index().pivot(index='genes', columns='channel', values=['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr'])

gene_mask = df_wide['avg_Pearson'].isna().any(axis=1)
df_wide = df_wide.loc[~gene_mask]
df_wide.to_csv(f'./results/all_channels_results_qc_clean.csv')
df_wide

avg_Pearson                        avg_Pearson_pval            \
channel FUCCI angle   Hoechst Segmentation      FUCCI angle   Hoechst   
genes                                                                   
Cald1      0.019108  0.103987     0.159037         0.697422  0.000001   

                     avg_Pearson_fdr                         
channel Segmentation     FUCCI angle   Hoechst Segmentation  
genes                                                        
Cald1            0.0        0.799154  0.000009          0.0